# Testing Navier Stokes solver.

1. Convert twoD -> threeD. DONE IN `TestingNavierStokes1.ipynb`
2. Add hydrostatic balance. DONE IN `TestingNavierStokes2.ipynb`
3. Add Coriolis force and customize to wind_forced_dishpan problem, except that the surface boundary condition is Dirichlet with a specified surface velocity. DONE IN `TestingNavierStokes3.ipynb`
4. Customize to stress boundary condition at upper surface, not specified velocity. Add Jacobian to solver.

4b. Add linear solve too, to test performance. Replace total pressure with pressure anomaly.

Based on: `gridap` Tutorial 8: Incompressible Navier-Stokes

twnh July '25

## Problem statement

The ultimate goal is to solve a nonlinear multi-field PDE. Consider here the lid-stress-driven flow for the incompressible rotating Navier-Stokes equations. Formally, the PDE we want to solve is: find the velocity vector $u$ and the pressure anomaly $p$ such that

$$
\left\lbrace
\begin{aligned}
-\nu  \nabla^2 u +  \frac{1}{\rho_0} \nabla p + f  \hat{\mathbf k} \times u = 0 &\text{ in }\Omega,\\
\nabla\cdot u = 0 &\text{ in } \Omega,\\
\boldsymbol{t} \cdot \boldsymbol{\sigma} \cdot \mathbf{n} = \tau_{\text{imposed}} &\text{ on } \Gamma_s, \\
u = 0 &\text{ on } \Gamma_w,
\end{aligned}
\right.
$$

where the computational domain is the rectangle $\Omega \doteq (0,L_x) \times (-L_y/2,L_y/2) \times (-L_z,0)$, ${\mathbf n}$ is the unit outward normal, and $\boldsymbol{t}$ is the tangential direction at the surface, $\Gamma_s$. The driving force is the tangential stress $\tau_{\text{imposed}}$ (units of $\text{m}^{2} \text{s}^{-2}$ ). The mean value of the pressure anomaly is constrained to equal zero,

$$
\int_\Omega p \ {\rm d}\Omega = 0, 
$$
and the total pressure is
$$
p_{\text{tot}} = p - \rho_0 g_r z .
$$

The weak form of this problem is (from GPT-4.1):

Find $(u,p,\lambda) \in V \times Q \times \mathbb{R}$ such that **for all** $(v, q, \mu) \in V \times Q \times \mathbb{R}$:

$$
\begin{aligned}
&\int_\Omega \nu \nabla u : \nabla v \, d\Omega
+ \int_\Omega (f\, \hat{\mathbf{k}} \times u) \cdot v\, d\Omega
- \frac{1}{\rho_0} \int_\Omega p\, \nabla \cdot v\, d\Omega 
+ \frac{1}{\rho_0} \int_\Omega q\, \nabla \cdot u\, d\Omega
+ \frac{1}{\rho_0} \int_\Omega \lambda\, q\, d\Omega
+ \frac{1}{\rho_0} \int_\Omega \mu\, p\, d\Omega 
= \int_{\Gamma_s} \tau_{\text{imposed}}\, (t \cdot v)\, dS
\end{aligned}
$$

Alternatively, by restricting the $p$ and $q$ fields to have zero mean by definition of their functions spaces, the weak form is:

$$
\boxed{
\begin{aligned}
&\int_\Omega \nu \nabla u : \nabla v \, d\Omega
+ \int_\Omega (f\, \hat{\mathbf{k}} \times u) \cdot v\, d\Omega
- \frac{1}{\rho_0} \int_\Omega p\, \nabla \cdot v\, d\Omega 
+ \frac{1}{\rho_0} \int_\Omega q\, \nabla \cdot u\, d\Omega
= \int_{\Gamma_s} \tau_{\text{imposed}}\, (t \cdot v)\, dS
\end{aligned}
}
$$

In [1]:
using Gridap

#### Define parameters

In [2]:
# The grid
Nx = 48             # number of points in x direction
Ny = 24             # number of points in y direction
Nz = 12             # number of points in the vertical direction

# The domain
Lx = 0.25            # (m) domain length
Ly = 0.125           # (m) domain width
Lz = 0.025           # (m) domain depth
domain = (0.0,Lx,-Ly/2,Ly/2,-Lz,0.0)
partition = (Nx,Ny,Nz)

# Create the mesh
model = CartesianDiscreteModel(domain,partition)

# Physical properties
# ν = 1e-6           # (m^2/s) kinematic viscosity
# ρₒ = 1000.0       # (kg/m^3) reference density
ν = 1.0e-1           # (m^2/s) kinematic viscosity
ρₒ = 1000.0       # (kg/m^3) reference density
# Rotation rate
rotation_period = 1500.0   # (s)
f = 2*2*π/rotation_period
# f=0.0
Ekman_layer_depth = sqrt(2*ν / f) # (m), Ekman layer depth
println("Ekman layer depth: ", Ekman_layer_depth, " m")

# Surface stress boundary condition
u₁₀(y) = 0.025     # m s⁻¹, average wind velocity 10 meters above the ocean
# u₁₀(y) = -0.25 + 0.25 .* cos(π.*y./Ly)     # m s⁻¹, average wind velocity 10 meters above the ocean
# u₁₀(y) =  0.25 .* cos(π.*y./Ly)     # m s⁻¹, average wind velocity 10 meters above the ocean
cᴰ = 2.5e-3 # dimensionless drag coefficient
ρₐ = 1.225  # kg m⁻³, average density of air at sea-level
Qᵘ(x) = VectorValue((ρₐ / ρₒ) * cᴰ * u₁₀(x[2]) * abs(u₁₀(x[2])),0,0) # m² s⁻²

Ekman layer depth: 4.8860251190292 m


Qᵘ (generic function with 1 method)

Create two new boundary tags,  namely `"lid"` and `"walls"` for the top side of the square (where the stress is applied), and for the rest of the boundary (where the velocity is zero).

In [3]:
labels = get_face_labeling(model)
add_tag_from_tags!(labels,"lid",[22,])
add_tag_from_tags!(labels,"walls",[1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,23,24,25,26]) ;

## FE spaces

For the velocities, create a conventional vector-valued continuous Lagrangian FE space. In this example, we select a second order interpolation.

In [4]:
order = 2
reffeᵤ = ReferenceFE(lagrangian,VectorValue{3,Float64},order)
V = TestFESpace(model,reffeᵤ,conformity=:H1,labels=labels,dirichlet_tags="walls")

UnconstrainedFESpace()

The interpolation space for the pressure anomaly is built as follows

In [5]:
reffeₚ = ReferenceFE(lagrangian,Float64,order-1;space=:P)
Q = TestFESpace(model,reffeₚ,conformity=:L2,constraint=:zeromean)

ZeroMeanFESpace()

With the options `:Lagrangian`, `space=:P`, `valuetype=Float64`, and `order=order-1`, we select the local polynomial space $P_{k-1}(T)$ on the cells $T\in\mathcal{T}$. With the symbol `space=:P` we specifically chose a local Lagrangian interpolation of type "P". Without using `space=:P`, would lead to a local Lagrangian of type "Q" since this is the default for quadrilateral or hexahedral elements. On the other hand, `constraint=:zeromean` leads to a FE space, whose functions are constrained to have mean value equal to zero, which is just what we need for the pressure anomaly space. With these objects, we build the trial multi-field FE spaces

In [6]:
uD0 = VectorValue(0,0,0)            # Velocity vanishes on the walls
U = TrialFESpace(V,uD0)
P = TrialFESpace(Q)
Y = MultiFieldFESpace([V, Q])
X = MultiFieldFESpace([U, P])

MultiFieldFESpace()

## Triangulation and integration quadrature

From the discrete model we can define the triangulation and integration measure

In [7]:
degree = order
Ωₕ = Triangulation(model)
dΩ = Measure(Ωₕ,degree)
Γ = BoundaryTriangulation(model,tags="lid")
dΓ = Measure(Γ,degree)

GenericMeasure()

The bilinear form reads

In [8]:
using LinearAlgebra
khat = VectorValue(0,0,1)
a((u,p),(v,q)) = ∫( ν*∇(v)⊙∇(u) - (1/ρₒ)*p*(∇⋅v) + (1/ρₒ)*q*(∇⋅u) + f*(cross(khat, u)⋅v))dΩ
b((v,q)) =  ∫(- v ⋅ Qᵘ)dΓ 

b (generic function with 1 method)

Finally, the Navier-Stokes weak form residual and Jacobian are defined as

In [9]:
res((u,p),(v,q)) = a((u,p),(v,q)) + b((v,q))
jac((u,p),(du,dp),(v,q)) = a((du,dp),(v,q)) 

jac (generic function with 1 method)

With the function `res` representing the weak residual, we build the nonlinear FE problem. Note that this problem is actually linear, but we anticipate nonlinear problems soon...

In [10]:
op = FEOperator(res,jac,X,Y)

FEOperatorFromWeakForm()

First guess solution is random. This is used by the nonlinear solver if that's called first.

In [11]:
import Random
Random.seed!(1234)
using Gridap.MultiField

# Generate random data for the free degrees of freedom
x_U = rand(Float64, num_free_dofs(U))
U0 = FEFunction(U, x_U)
x_P = rand(Float64, num_free_dofs(P))
P0 = FEFunction(P, x_P)

# Create a MultiFieldFESpace from U and P
multi_field_space = MultiFieldFESpace([U, P])

# Create the MultiFieldFEFunction using the three arguments required by the constructor
X0 = MultiFieldFEFunction([x_U; x_P], multi_field_space, [U0, P0]) ;

## Linear solver phase

Try linear solver first

In [12]:
using GridapSolvers
op_l = AffineFEOperator(a,b,X,Y)
# ls_l = LUSolver()
ls_l = GMRESSolver(4)

solver_l = LinearFESolver(ls_l)
X0_l = X0

if @isdefined(cache_l)
    @time X0_l,cache_l = solve!(X0_l,solver_l,op_l,cache_l)
else
    @time X0_l,cache_l = solve!(X0_l,solver_l,op_l)
end

# Accessing the fields within the MultiFieldFEFunction
uh_l = X0_l.single_fe_functions[1]
ph_l = X0_l.single_fe_functions[2] 
writevtk(Ωₕ,"TestingNavierStokes4b_l",cellfields=["uh"=>uh_l,"ph"=>ph_l])

  4.877232 seconds (4.96 M allocations: 548.726 MiB, 2.86% gc time, 23.38% compilation time)


(["TestingNavierStokes4b_l.vtu"],)

## Nonlinear solver phase

In [13]:
using LineSearches: BackTracking
nls = NLSolver(show_trace=true, method=:newton, linesearch=BackTracking())
solver = FESolver(nls)

NonlinearFESolver()

Solve the problem with an initial guess from the linear solver if it's the first time, otherwise, use the previous solution. This allows parameter changes to be explored, but not different resolutions.

In [14]:
if(true)
if @isdefined(cache)
    @time X0,cache = solve!(X0,solver,op,cache)
else
    @time X0,cache = solve!(X0_l,solver,op)
end

# Accessing the fields within the MultiFieldFEFunction
uh = X0.single_fe_functions[1]
ph = X0.single_fe_functions[2] 
writevtk(Ωₕ,"TestingNavierStokes4b_nl",cellfields=["uh"=>uh,"ph"=>ph])
end

Iter     f(x) inf-norm    Step 2-norm 
------   --------------   --------------
     0     2.813125e-07              NaN
     1     5.907533e-12     1.861719e+04
328.625719 seconds (24.23 M allocations: 68.810 GiB, 0.47% gc time, 1.83% compilation time)


(["TestingNavierStokes4b_nl.vtu"],)